# Early baseline comparison: reproduction (Reddit, 1 h)

**Purpose: provenance.** This notebook re-runs the code of an earlier comparison table so that the origin of its values can be documented (see `PROVENANCE.md`): Triadic 0.4688 / 0.5852, "Temporal GNN" 0.5585 / 0.3682, "DSBM" 0.6769 / 0.3025 (Silhouette / Davies-Bouldin). **These rows are not used anywhere in the reported results**, which come from `01_pipeline.ipynb`.

The cells follow the original code without modification. The only plumbing change is that snapshots and triads are read from the per-frequency directory (`snapshots/freq_<FREQ>`) so that frequencies are not mixed.

What the three rows actually compute:
* **Triadic**: HDBSCAN(min_cluster_size=5, min_samples=2) on 6 standardised triadic-signature features (no neural model); noise points removed before scoring.
* **"Temporal GNN"**: an *untrained* 2-layer GCN (node feature = degree, mean-pooled, 32-d, torch seed 12345), one embedding per snapshot independently (no temporal component); HDBSCAN(min_cluster_size=10); noise removed.
* **"DSBM"**: per-snapshot Louvain modularity (one scalar, `randomize=False`); HDBSCAN(min_cluster_size=10); noise removed.

Set `DATASET` (`higgs` or `reddit`) and `FREQ` in the configuration cell. The original comparison used `FREQ = "1h"`.
The saved outputs below are from the Reddit, 1 h run.

In [1]:
# Setup (run once). torch_geometric core is enough for GCNConv/global_mean_pool.
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","hdbscan","python-louvain","torch_geometric"],check=False)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'hdbscan', 'python-louvain', 'torch_geometric'], returncode=0)

In [2]:
# ---- CONFIG ----
DATASET = "reddit"      # "higgs" or "reddit"
FREQ    = "1h"         # original GNN cell used '1h'; 6h/12h also generated in the originals
SEED    = 42

import os, json, math, csv, random, logging, warnings
from pathlib import Path
from collections import OrderedDict
import numpy as np, pandas as pd, networkx as nx
from tqdm.auto import tqdm
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED)

ROOT = Path("/content") / f"early_{DATASET}_{FREQ}"
ROOT.mkdir(parents=True, exist_ok=True)
SNAP_DIR = ROOT / "snapshots" / f"freq_{FREQ}"
TRIADS_CSV = ROOT / "triads_ordered.csv"
SIGNATURES_CSV = ROOT / "triadic_signatures.csv"
PUBLISHED = {"Triadic":(0.4688,0.5852), "Temporal GNN":(0.5585,0.3682), "DSBM":(0.6769,0.3025)}   # values of the earlier comparison table

In [3]:
# ---- DATA + SNAPSHOTS (original cells: Higgs 'DATA & SNAPSHOTS'; Reddit loader cell) ----
import requests
def download(dest, url):
    dest = Path(dest)
    if dest.exists(): return dest
    r = requests.get(url, stream=True); r.raise_for_status()
    with open(dest,"wb") as fh:
        for ch in r.iter_content(1024*1024):
            if ch: fh.write(ch)
    return dest

def load_higgs():
    p = download(ROOT/"higgs-activity_time.txt.gz","https://snap.stanford.edu/data/higgs-activity_time.txt.gz")
    df = pd.read_csv(p, compression="gzip", sep=r"\s+", header=None, names=["src","dst","ts","interaction"],
                     engine="c", dtype={"src":int,"dst":int,"ts":float,"interaction":str})
    unit = 'ms' if df['ts'].median() > 1e12 else 's'
    df['tstamp'] = pd.to_datetime(df['ts'], unit=unit)
    return df.sort_values('tstamp').reset_index(drop=True)

def load_reddit():
    p = download(ROOT/"reddit_activity_time.tsv","https://snap.stanford.edu/data/soc-redditHyperlinks-body.tsv")
    df = pd.read_csv(p, sep="\t", header=0, engine="python",
                     dtype={"SOURCE_SUBREDDIT":str,"TARGET_SUBREDDIT":str,"TIMESTAMP":str,"LINK_SENTIMENT":int,"PROPERTIES":str})
    df = df.rename(columns={"SOURCE_SUBREDDIT":"src","TARGET_SUBREDDIT":"dst","TIMESTAMP":"ts","LINK_SENTIMENT":"interaction"})
    df['tstamp'] = pd.to_datetime(df['ts'])
    return df.sort_values('tstamp').reset_index(drop=True)

def make_snapshots(df, freq, out_dir, directed=True, weight_attr='weight', int_nodes=True):
    outp = Path(out_dir); outp.mkdir(parents=True, exist_ok=True)
    windows = df['tstamp'].dt.floor(freq).sort_values().unique()
    floor_col = df['tstamp'].dt.floor(freq)
    saved = []
    for w in tqdm(windows, desc=f'snapshots ({freq})'):
        st = w
        gdf = df.loc[floor_col == w]
        if gdf.empty: continue
        edges = gdf.groupby(['src','dst']).size().reset_index(name=weight_attr)
        G = nx.DiGraph() if directed else nx.Graph()
        nodes = pd.unique(gdf[['src','dst']].values.ravel())
        G.add_nodes_from([int(n) if int_nodes else n for n in nodes])
        for _, r in edges.iterrows():
            a, b = (int(r['src']), int(r['dst'])) if int_nodes else (r['src'], r['dst'])
            G.add_edge(a, b, weight=int(r[weight_attr]))
        name = str(st).replace(":", "-")
        nx.write_graphml(G, str(outp / f"snapshot_{name}.graphml"))
        saved.append((st, name))
    logging.info(f"Snapshots written for freq={freq}: {len(saved)} to {outp}")
    return saved

if 'df_activity' not in globals():     # allows injecting a test frame
    df_activity = load_higgs() if DATASET=="higgs" else load_reddit()
logging.info(f"{DATASET}: {len(df_activity)} interactions, {df_activity['tstamp'].min()} -> {df_activity['tstamp'].max()}")
if not list(SNAP_DIR.glob("snapshot_*.graphml")):
    make_snapshots(df_activity, FREQ, SNAP_DIR, int_nodes=(DATASET=="higgs"))
N_SNAP_FILES = len(list(SNAP_DIR.glob("snapshot_*.graphml")))
print("snapshot files:", N_SNAP_FILES)

snapshots (1h):   0%|          | 0/28979 [00:00<?, ?it/s]

snapshot files: 28979


In [4]:
# ---- TRIAD EXTRACTION (original 'FAST TRIAD EXTRACTOR', full enumeration branch, neighbour caps unchanged) ----
SAMPLE_MAX_TRIADS_PER_SNAP = None
MAX_NEIGHBORS = 800
NEIGHBOR_SAMPLE_IF_BIG = 400

def triad_bits(G, a,b,c):
    return [1 if G.has_edge(a,b) else 0, 1 if G.has_edge(b,a) else 0,
            1 if G.has_edge(a,c) else 0, 1 if G.has_edge(c,a) else 0,
            1 if G.has_edge(b,c) else 0, 1 if G.has_edge(c,b) else 0]
def bits_to_int(bits):
    v = 0
    for i,bit in enumerate(bits[::-1]): v |= (bit << i)
    return v
def micro_to_macro(bits):
    ab = bits[0] or bits[1]; ac = bits[2] or bits[3]; bc = bits[4] or bits[5]
    und = ab + ac + bc
    recip = (bits[0] and bits[1]) + (bits[2] and bits[3]) + (bits[4] and bits[5])
    if und <= 1: return "S1_Open"
    if und == 2 and recip == 0: return "S2_Feedforward"
    if und == 2 and recip == 1: return "S3_WeakReciprocal"
    if und == 2 and recip >= 2: return "S4_StrongReciprocal"
    if und == 3 and recip == 0: return "S5_ClosedTriangle"
    return "S6_FullyConnected"

files = sorted(SNAP_DIR.glob("snapshot_*.graphml"))
if TRIADS_CSV.exists(): TRIADS_CSV.unlink()
random.seed(SEED); np.random.seed(SEED)
with open(TRIADS_CSV, "a", newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(["window","A","B","C","microstate_id","microstate_bits","macrostate"])
    for fpath in tqdm(files, desc="triads"):
        label_raw = fpath.name.replace("snapshot_","").replace(".graphml","")
        try: wlabel = pd.to_datetime(label_raw)
        except Exception: wlabel = label_raw
        G = nx.read_graphml(str(fpath))
        try: G = nx.relabel_nodes(G, {n:int(n) for n in G})
        except Exception: pass
        if not G.is_directed(): G = G.to_directed()
        for v in G.nodes():
            nbrs = list(set(list(G.successors(v)) + list(G.predecessors(v))))
            if MAX_NEIGHBORS is not None and len(nbrs) > MAX_NEIGHBORS:
                nbrs = list(np.random.choice(nbrs, size=min(NEIGHBOR_SAMPLE_IF_BIG, len(nbrs)), replace=False))
            if len(nbrs) < 2: continue
            ln = len(nbrs)
            for i in range(ln):
                for j in range(i+1, ln):
                    trip = sorted([v, nbrs[i], nbrs[j]])
                    bits = triad_bits(G, *trip)
                    writer.writerow([str(wlabel), trip[0], trip[1], trip[2], bits_to_int(bits), ''.join(map(str,bits)), micro_to_macro(bits)])
logging.info("triads written")

triads:   0%|          | 0/28979 [00:00<?, ?it/s]

In [5]:
# ---- SIGNATURES (original signature cell, logic unchanged; sequential instead of multiprocessing Pool) ----
df_triads = pd.read_csv(TRIADS_CSV, dtype={'window':str,'microstate_id':float,'microstate_bits':str,'macrostate':str})
macro_vocab = OrderedDict(sorted({m: None for m in df_triads['macrostate'].dropna().unique()}.items()))
macros = list(macro_vocab.keys())
def entropy_from_counts(counts):
    s = sum(counts)
    if s == 0: return 0.0
    p = np.array(counts, dtype=float)/s; p = p[p>0]
    return -(p*np.log2(p)).sum()
def parse_and_calc_reciprocity(bits_s):
    if pd.isna(bits_s): return 0.0
    b = [int(ch) for ch in str(bits_s)]
    return ((1 if (b[0] and b[1]) else 0)+(1 if (b[2] and b[3]) else 0)+(1 if (b[4] and b[5]) else 0))/3.0
def process_single_window(w, g):
    total = len(g)
    cm = g['macrostate'].value_counts().to_dict()
    Mvec = [cm.get(m,0) for m in macros]
    mc = g['microstate_id'].dropna().astype(int).value_counts().to_dict()
    micro_list = [mc.get(k,0) for k in sorted(mc.keys())] if mc else []
    recips = g['microstate_bits'].apply(parse_and_calc_reciprocity).tolist()
    macro_ent = entropy_from_counts(Mvec); micro_ent = entropy_from_counts(micro_list) if micro_list else 0.0
    closed_states = [m for m in macros if ('5' in m or '6' in m or 'closed' in m.lower())]
    closed = sum(cm.get(m,0) for m in closed_states)
    return {'window':w,'triad_count':total,'closure_intensity':closed/total if total>0 else 0.0,
            'avg_reciprocity':float(np.mean(recips)) if recips else 0.0,'macro_entropy':macro_ent,'micro_entropy':micro_ent,
            'volatility':macro_ent/math.log2(max(2,total)) if total>0 else 0.0,
            'macro_counts':json.dumps({m:cm.get(m,0) for m in macros})}
rows = [process_single_window(w,g) for w,g in tqdm(df_triads.groupby('window'), desc="signatures")]
sig_df = pd.DataFrame(rows).sort_values('window').reset_index(drop=True)
sig_df.to_csv(SIGNATURES_CSV, index=False)
print(len(sig_df), "signature windows")

signatures:   0%|          | 0/15565 [00:00<?, ?it/s]

15565 signature windows


In [6]:
# ---- ROW 1: "Triadic" — HDBSCAN on 6 standardized signature features, noise removed (original HDBSCAN + evaluation cells) ----
import hdbscan
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
df = pd.read_csv(SIGNATURES_CSV)
features = ['closure_intensity','avg_reciprocity','macro_entropy','micro_entropy','volatility','triad_count']
Xs = StandardScaler().fit_transform(df[features].fillna(0).values)
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=2, prediction_data=True); clusterer.fit(Xs)
df['hdbscan_state'] = clusterer.labels_
m = (df['hdbscan_state'] != -1); Xs_filtered, labels_filtered = Xs[m], df['hdbscan_state'][m].values
results = {}
def record(name, X, labels, n_total):
    n_noise = int((labels==-1).sum()) if hasattr(labels,'__len__') else 0
    ok = len(np.unique(labels[labels!=-1]))>=2
    results[name] = {"silhouette": float(silhouette_score(X[labels!=-1],labels[labels!=-1])) if ok else np.nan,
                     "davies_bouldin": float(davies_bouldin_score(X[labels!=-1],labels[labels!=-1])) if ok else np.nan,
                     "n_points": int(n_total), "n_clusters": int(len(set(labels))-(1 if -1 in labels else 0)), "n_noise": n_noise}
record("Triadic", Xs, clusterer.labels_, len(Xs))
print(results["Triadic"])

{'silhouette': 0.9606413875713355, 'davies_bouldin': 0.4815381988617936, 'n_points': 15565, 'n_clusters': 208, 'n_noise': 398}


In [7]:
# ---- ROW 2: "Temporal GNN" — UNTRAINED 2-layer GCN on degree, mean-pooled (original GNN cells) ----
import torch, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
gnn_data_list = []
for fpath in tqdm(sorted(SNAP_DIR.glob("snapshot_*.graphml")), desc="to PyG"):
    G = nx.read_graphml(str(fpath))
    if not G.nodes(): continue
    mapping = {old:new for new,old in enumerate(list(G.nodes()))}
    Gr = nx.relabel_nodes(G, mapping, copy=True)
    x = torch.tensor(np.array([d for n,d in Gr.degree()]), dtype=torch.float).view(-1,1)
    s,t,w = [],[],[]
    for u,v,d in Gr.edges(data=True): s.append(u); t.append(v); w.append(d.get('weight',1.0))
    if s: ei = torch.tensor([s,t],dtype=torch.long); ea = torch.tensor(w,dtype=torch.float).view(-1,1)
    else: ei = torch.empty((2,0),dtype=torch.long); ea = torch.empty((0,1),dtype=torch.float)
    gnn_data_list.append(Data(x=x, edge_index=ei, edge_attr=ea))

class SimpleGNN(torch.nn.Module):
    def __init__(self, nf, hc, ngf):
        super().__init__(); torch.manual_seed(12345)
        self.conv1 = GCNConv(nf, hc); self.conv2 = GCNConv(hc, hc); self.lin = torch.nn.Linear(hc, ngf)
    def forward(self, data):
        x, ei, b = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, ei)); x = F.relu(self.conv2(x, ei))
        return self.lin(global_mean_pool(x, b))
model = SimpleGNN(1, 64, 32); model.eval()      # NO TRAINING, as in the original
emb = []
with torch.no_grad():
    for d in tqdm(gnn_data_list, desc="GNN embeddings"):
        if d.batch is None: d.batch = torch.zeros(d.x.size(0), dtype=torch.long)
        emb.append(model(d).squeeze().numpy())
gnn_embeddings = np.array(emb)
gl = hdbscan.HDBSCAN(min_cluster_size=10, prediction_data=True).fit_predict(gnn_embeddings)
record("Temporal GNN", gnn_embeddings, gl, len(gnn_embeddings))
print(results["Temporal GNN"])

to PyG:   0%|          | 0/28979 [00:00<?, ?it/s]

GNN embeddings:   0%|          | 0/28979 [00:00<?, ?it/s]

{'silhouette': 0.890514612197876, 'davies_bouldin': 3.1082214882214494, 'n_points': 28979, 'n_clusters': 485, 'n_noise': 2575}


In [8]:
# ---- ROW 3: "DSBM" — per-snapshot Louvain modularity (1 scalar) (original DSBM-proxy cell) ----
import community.community_louvain as co_louvain
mods, wins = [], []
for fpath in tqdm(sorted(SNAP_DIR.glob("snapshot_*.graphml")), desc="Louvain"):
    G = nx.read_graphml(str(fpath))
    if not G.nodes(): continue
    try: G = nx.relabel_nodes(G, {n:int(n) for n in G})
    except ValueError: pass
    Gu = G.to_undirected()
    part = co_louvain.best_partition(Gu, randomize=False)
    mods.append(co_louvain.modularity(part, Gu)); wins.append(fpath.name)
df_dsbm = pd.DataFrame({'window':wins,'modularity':mods}).dropna().reset_index(drop=True)
Xd = np.array(df_dsbm[['modularity']].values, dtype=float)   # writable copy (newer pandas returns read-only views; hdbscan needs writable)
dl = hdbscan.HDBSCAN(min_cluster_size=10, prediction_data=True).fit_predict(Xd)
record("DSBM", Xd, dl, len(Xd))
print(results["DSBM"])

Louvain:   0%|          | 0/28979 [00:00<?, ?it/s]

{'silhouette': 0.5028334667733927, 'davies_bouldin': 0.17675508401686274, 'n_points': 28979, 'n_clusters': 256, 'n_noise': 862}


In [9]:
# ---- RESULTS + comparison with the values of the earlier table ----
out = pd.DataFrame(results).T
out["published_silhouette"] = [PUBLISHED[k][0] for k in out.index]
out["published_db"] = [PUBLISHED[k][1] for k in out.index]
out.insert(0,"dataset",DATASET); out.insert(1,"freq",FREQ)
display(out)
out.to_csv(ROOT/f"early_comparison_results_{DATASET}_{FREQ}.csv")
import shutil; shutil.make_archive(f"/content/early_comparison_{DATASET}_{FREQ}", "zip", ROOT, ".")   # includes signatures CSV + results
print("saved; download /content/early_comparison_%s_%s.zip" % (DATASET,FREQ))

,dataset,freq,silhouette,davies_bouldin,n_points,n_clusters,n_noise,published_silhouette,published_db
Triadic,reddit,1h,0.960641,0.481538,15565.0,208.0,398.0,0.4688,0.5852
Temporal GNN,reddit,1h,0.890515,3.108221,28979.0,485.0,2575.0,0.5585,0.3682
DSBM,reddit,1h,0.502833,0.176755,28979.0,256.0,862.0,0.6769,0.3025


saved; download /content/early_comparison_reddit_1h.zip
